In [1]:
"""
Enhanced Grazioso Salvare Animal Shelter Dashboard

CS 499 Milestone Three:
Algorithms and Data Structures Enhancement

Enhancements:
1. Loads MongoDB data once and stores it in a reusable pandas DataFrame.
2. Uses a dictionary for constant-time rescue-category lookup.
3. Uses vectorized pandas filtering rather than repeated database queries.
4. Uses immutable tuples for rescue breed criteria.
5. Applies stable sorting by breed and animal name.
6. Uses named columns instead of positional DataFrame indexes.
7. Handles empty results, missing values, and invalid coordinates safely.
"""
import os
import sys

PROJECT_FOLDER = "/home/codio/workspace/code_files/enhanced"

os.chdir(PROJECT_FOLDER)

if PROJECT_FOLDER not in sys.path:
    sys.path.insert(0, PROJECT_FOLDER)

print("Working directory:", os.getcwd())
print("Project folder added to Python path.")

import logging
import re

from jupyter_dash import JupyterDash
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import pandas as pd
import plotly.express as px

from animal_shelter import AnimalShelter
from config import USERNAME, PASSWORD


# ---------------------------------------------------------
# Logging Configuration
# ---------------------------------------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

LOGGER = logging.getLogger(__name__)


# ---------------------------------------------------------
# Database Connection
# ---------------------------------------------------------

try:
    shelter = AnimalShelter(USERNAME, PASSWORD)
except Exception as error:
    LOGGER.exception("The dashboard could not connect to MongoDB.")
    raise RuntimeError(
        "Unable to start the dashboard because the MongoDB connection failed."
    ) from error


# ---------------------------------------------------------
# Algorithms and Data Structures
# ---------------------------------------------------------

# Dictionary lookup provides direct access to each category's criteria.
# Tuples are used because the breed collections should not change while
# the dashboard is running.
RESCUE_CRITERIA = {
    "water": {
        "breeds": (
            "Labrador Retriever",
            "Chesapeake Bay Retriever",
            "Newfoundland",
        ),
        "sex": "Intact Female",
        "label": "Water Rescue",
    },
    "mountain": {
        "breeds": (
            "German Shepherd",
            "Alaskan Malamute",
            "Old English Sheepdog",
            "Siberian Husky",
            "Rottweiler",
        ),
        "sex": "Intact Male",
        "label": "Mountain or Wilderness Rescue",
    },
    "disaster": {
        "breeds": (
            "Doberman Pinscher",
            "German Shepherd",
            "Golden Retriever",
            "Bloodhound",
            "Rottweiler",
        ),
        "sex": "Intact Male",
        "label": "Disaster or Individual Tracking",
    },
}


def load_master_dataframe():
    """
    Load animal records from MongoDB once.

    The resulting DataFrame becomes the master in-memory data structure
    reused by dashboard callbacks. This avoids querying MongoDB every
    time the user selects a filter.
    """

    try:
        records = shelter.read(
            query={},
            sort=[
                ("breed", 1),
                ("name", 1),
            ],
        )

        dataframe = pd.DataFrame.from_records(records)

        if dataframe.empty:
            LOGGER.warning("MongoDB returned no animal records.")
            return dataframe

        # Replace missing values in text columns to prevent filtering errors.
        text_columns = [
            "animal_id",
            "animal_type",
            "breed",
            "name",
            "sex_upon_outcome",
            "outcome_type",
        ]

        for column in text_columns:
            if column in dataframe.columns:
                dataframe[column] = dataframe[column].fillna("").astype(str)

        # Convert coordinate fields to numeric values.
        for coordinate_column in ["location_lat", "location_long"]:
            if coordinate_column in dataframe.columns:
                dataframe[coordinate_column] = pd.to_numeric(
                    dataframe[coordinate_column],
                    errors="coerce",
                )

        # Stable sorting gives predictable output when values are equal.
        sort_columns = [
            column
            for column in ["breed", "name", "animal_id"]
            if column in dataframe.columns
        ]

        if sort_columns:
            dataframe = dataframe.sort_values(
                by=sort_columns,
                kind="mergesort",
                na_position="last",
            )

        dataframe = dataframe.reset_index(drop=True)

        LOGGER.info(
            "Loaded %d animal records into the master DataFrame.",
            len(dataframe),
        )

        return dataframe

    except Exception:
        LOGGER.exception("Animal records could not be loaded.")
        return pd.DataFrame()


# Load data once instead of inside every callback.
MASTER_DF = load_master_dataframe()


def build_breed_pattern(breeds):
    """
    Build a safe regular-expression pattern from breed keywords.

    re.escape prevents special characters in breed names from being
    interpreted as regular-expression operators.
    """

    escaped_breeds = [re.escape(breed) for breed in breeds]
    return "|".join(escaped_breeds)


def filter_animals(filter_type):
    """
    Filter the master DataFrame using vectorized pandas operations.

    Complexity:
    - Dictionary criteria lookup: average O(1)
    - Vectorized filtering: O(n)
    - Sorting filtered records: O(k log k)

    Here, n is the number of records in the master DataFrame and k is
    the number of matching records.
    """

    if MASTER_DF.empty:
        return MASTER_DF.copy()

    # Reset returns all records without additional filtering.
    if filter_type == "reset":
        filtered_df = MASTER_DF.copy()

    else:
        # Dictionary lookup avoids a long if/elif chain.
        criteria = RESCUE_CRITERIA.get(filter_type)

        if criteria is None:
            LOGGER.warning(
                "Unknown filter '%s'. Showing all records.",
                filter_type,
            )
            filtered_df = MASTER_DF.copy()

        else:
            if (
                "breed" not in MASTER_DF.columns
                or "sex_upon_outcome" not in MASTER_DF.columns
            ):
                LOGGER.error(
                    "Required filtering columns are missing."
                )
                return MASTER_DF.iloc[0:0].copy()

            breed_pattern = build_breed_pattern(criteria["breeds"])

            # Vectorized string comparison checks all rows efficiently.
            breed_mask = MASTER_DF["breed"].str.contains(
                breed_pattern,
                case=False,
                na=False,
                regex=True,
            )

            sex_mask = MASTER_DF[
                "sex_upon_outcome"
            ].str.casefold().eq(
                criteria["sex"].casefold()
            )

            filtered_df = MASTER_DF.loc[
                breed_mask & sex_mask
            ].copy()

    # Sort results consistently for easier review.
    sort_columns = [
        column
        for column in ["breed", "name", "animal_id"]
        if column in filtered_df.columns
    ]

    if sort_columns:
        filtered_df = filtered_df.sort_values(
            by=sort_columns,
            kind="mergesort",
            na_position="last",
        )

    return filtered_df.reset_index(drop=True)


def get_filter_label(filter_type):
    """Return a user-friendly label for the selected filter."""

    if filter_type == "reset":
        return "All Animals"

    criteria = RESCUE_CRITERIA.get(filter_type)

    if criteria is None:
        return "All Animals"

    return criteria["label"]


def create_default_map(message="Select an animal with valid coordinates."):
    """Create a map centered near Austin, Texas, without a marker."""

    return dl.Map(
        style={
            "width": "100%",
            "height": "500px",
        },
        center=[30.75, -97.48],
        zoom=9,
        children=[
            dl.TileLayer(),
            dl.ScaleControl(position="bottomleft"),
            dl.Marker(
                position=[30.75, -97.48],
                children=[
                    dl.Tooltip(message),
                ],
            ),
        ],
    )


# ---------------------------------------------------------
# Dashboard Layout
# ---------------------------------------------------------

app = JupyterDash("ProjectTwo")

table_columns = [
    {
        "name": column,
        "id": column,
        "deletable": False,
        "selectable": True,
    }
    for column in MASTER_DF.columns
]

app.layout = html.Div(
    [
        html.Center(
            html.B(
                html.H1("Grazioso Salvare Dashboard")
            )
        ),

        html.Center(
            html.H4(
                "Praise Oulare - CS 499 Algorithms and Data Structures Enhancement"
            )
        ),

        html.Hr(),

        html.Div(
            [
                dcc.RadioItems(
                    id="filter-type",
                    options=[
                        {
                            "label": "Water Rescue",
                            "value": "water",
                        },
                        {
                            "label": "Mountain or Wilderness Rescue",
                            "value": "mountain",
                        },
                        {
                            "label": "Disaster or Individual Tracking",
                            "value": "disaster",
                        },
                        {
                            "label": "Reset (Show All)",
                            "value": "reset",
                        },
                    ],
                    value="reset",
                    labelStyle={
                        "display": "inline-block",
                        "padding": "10px",
                    },
                )
            ],
            style={"textAlign": "center"},
        ),

        html.Div(
            id="filter-summary",
            style={
                "textAlign": "center",
                "fontWeight": "bold",
                "padding": "10px",
            },
        ),

        html.Hr(),

        dash_table.DataTable(
            id="datatable-id",
            columns=table_columns,
            data=MASTER_DF.to_dict("records"),
            page_size=10,
            sort_action="native",
            sort_mode="multi",
            filter_action="native",
            row_selectable="single",
            selected_rows=[],
            style_table={
                "overflowX": "auto",
            },
            style_cell={
                "textAlign": "left",
                "minWidth": "100px",
                "width": "150px",
                "maxWidth": "250px",
                "whiteSpace": "normal",
            },
            style_header={
                "fontWeight": "bold",
            },
        ),

        html.Br(),

        html.Div(
            className="row",
            children=[
                html.Div(
                    id="graph-id",
                    className="col s12 m6",
                    style={
                        "width": "50%",
                        "display": "inline-block",
                        "verticalAlign": "top",
                    },
                ),
                html.Div(
                    id="map-id",
                    className="col s12 m6",
                    style={
                        "width": "50%",
                        "display": "inline-block",
                        "verticalAlign": "top",
                    },
                ),
            ],
        ),
    ]
)


# ---------------------------------------------------------
# Dashboard Callbacks
# ---------------------------------------------------------

@app.callback(
    [
        Output("datatable-id", "data"),
        Output("datatable-id", "selected_rows"),
        Output("filter-summary", "children"),
    ],
    [Input("filter-type", "value")],
)
def update_dashboard(filter_type):
    """
    Filter the in-memory DataFrame and update the table.

    MongoDB is not queried during this callback.
    """

    filtered_df = filter_animals(filter_type)
    filter_label = get_filter_label(filter_type)

    record_count = len(filtered_df)

    if record_count == 1:
        summary = f"{filter_label}: 1 matching animal"
    else:
        summary = (
            f"{filter_label}: {record_count:,} matching animals"
        )

    LOGGER.info(
        "Filter '%s' returned %d records.",
        filter_type,
        record_count,
    )

    return (
        filtered_df.to_dict("records"),
        [],
        summary,
    )


@app.callback(
    [
        Output("map-id", "children"),
        Output("graph-id", "children"),
    ],
    [
        Input("datatable-id", "derived_virtual_data"),
        Input(
            "datatable-id",
            "derived_virtual_selected_rows",
        ),
    ],
)
def update_map_and_graph(view_data, selected_rows):
    """
    Update the map and breed chart using the table's current data.

    The function safely handles filtering, sorting, missing records,
    and records without valid geographic coordinates.
    """

    if not view_data:
        empty_chart = px.pie(
            names=[],
            values=[],
            title="Breed Distribution - No Records Available",
        )

        empty_chart.update_layout(
            annotations=[
                {
                    "text": "No records match the current filters.",
                    "showarrow": False,
                    "font": {"size": 16},
                }
            ]
        )

        return (
            [create_default_map("No animal records are available.")],
            [dcc.Graph(figure=empty_chart)],
        )

    current_df = pd.DataFrame.from_records(view_data)

    # -----------------------------------------------------
    # Pie Chart Algorithm
    # -----------------------------------------------------

    if "breed" in current_df.columns:
        breed_counts = (
            current_df["breed"]
            .fillna("Unknown Breed")
            .replace("", "Unknown Breed")
            .value_counts()
            .rename_axis("breed")
            .reset_index(name="count")
        )

        # Show the ten most common breeds to keep the chart readable.
        top_breeds = breed_counts.head(10)

        figure = px.pie(
            top_breeds,
            names="breed",
            values="count",
            title="Top 10 Breed Distribution",
        )

        figure.update_traces(
            textposition="inside",
            textinfo="percent+label",
        )

    else:
        figure = px.pie(
            names=[],
            values=[],
            title="Breed Distribution",
        )

    graph_view = [dcc.Graph(figure=figure)]

    # -----------------------------------------------------
    # Map Algorithm
    # -----------------------------------------------------

    required_map_columns = {
        "location_lat",
        "location_long",
    }

    if not required_map_columns.issubset(
        current_df.columns
    ):
        return (
            [
                create_default_map(
                    "Location columns are unavailable."
                )
            ],
            graph_view,
        )

    if selected_rows:
        selected_index = selected_rows[0]
    else:
        selected_index = 0

    if (
        selected_index < 0
        or selected_index >= len(current_df)
    ):
        selected_index = 0

    selected_animal = current_df.iloc[selected_index]

    latitude = pd.to_numeric(
        selected_animal.get("location_lat"),
        errors="coerce",
    )

    longitude = pd.to_numeric(
        selected_animal.get("location_long"),
        errors="coerce",
    )

    if pd.isna(latitude) or pd.isna(longitude):
        return (
            [
                create_default_map(
                    "The selected animal does not have valid coordinates."
                )
            ],
            graph_view,
        )

    animal_name = selected_animal.get(
        "name",
        "Unknown",
    )

    if pd.isna(animal_name) or str(animal_name).strip() == "":
        animal_name = "Unknown"

    animal_breed = selected_animal.get(
        "breed",
        "Unknown Breed",
    )

    if pd.isna(animal_breed) or str(animal_breed).strip() == "":
        animal_breed = "Unknown Breed"

    animal_id = selected_animal.get(
        "animal_id",
        "Unknown",
    )

    map_view = [
        dl.Map(
            style={
                "width": "100%",
                "height": "500px",
            },
            center=[
                float(latitude),
                float(longitude),
            ],
            zoom=12,
            children=[
                dl.TileLayer(),
                dl.ScaleControl(
                    position="bottomleft"
                ),
                dl.Marker(
                    position=[
                        float(latitude),
                        float(longitude),
                    ],
                    children=[
                        dl.Tooltip(
                            f"{animal_name} - {animal_breed}"
                        ),
                        dl.Popup(
                            [
                                html.H3(str(animal_name)),
                                html.P(
                                    f"Animal ID: {animal_id}"
                                ),
                                html.P(
                                    f"Breed: {animal_breed}"
                                ),
                            ]
                        ),
                    ],
                ),
            ],
        )
    ]

    return map_view, graph_view


# ---------------------------------------------------------
# Run Dashboard
# ---------------------------------------------------------

if __name__ == "__main__":
    app.run_server(
        mode="inline",
        debug=False,
    )

Working directory: /home/codio/workspace/code_files/enhanced
Project folder added to Python path.


2026-07-27 06:22:01,057 - ERROR - Unable to connect to MongoDB.
Traceback (most recent call last):
  File "/home/codio/workspace/code_files/enhanced/animal_shelter.py", line 105, in __init__
    self.client.admin.command("ping")
  File "/home/codio/.pyenv/versions/3.11.2/lib/python3.11/site-packages/pymongo/_csot.py", line 125, in csot_wrapper
    return func(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/codio/.pyenv/versions/3.11.2/lib/python3.11/site-packages/pymongo/synchronous/database.py", line 926, in command
    with self._client._conn_for_reads(read_preference, session, operation=command_name) as (
  File "/home/codio/.pyenv/versions/3.11.2/lib/python3.11/contextlib.py", line 137, in __enter__
    return next(self.gen)
           ^^^^^^^^^^^^^^
  File "/home/codio/.pyenv/versions/3.11.2/lib/python3.11/site-packages/pymongo/synchronous/mongo_client.py", line 1846, in _conn_from_server
    with self._checkout(server, session) as conn:
  File "/home/c

RuntimeError: Unable to start the dashboard because the MongoDB connection failed.

In [ ]:
import os
import sys

PROJECT_FOLDER = "/home/codio/workspace/code_files/enhanced"

os.chdir(PROJECT_FOLDER)

if PROJECT_FOLDER not in sys.path:
    sys.path.insert(0, PROJECT_FOLDER)

print("Current directory:", os.getcwd())
print("config.py exists:", os.path.exists("config.py"))
print("animal_shelter.py exists:", os.path.exists("animal_shelter.py"))

In [ ]:
from config import USERNAME, PASSWORD
from animal_shelter import AnimalShelter

print(USERNAME)
print("Imports successful")